# 05 Train LaBraM and compare runs

Train on raw vs Zuna-augmented epochs and compare results.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "scripts").exists():
    if (ROOT.parent / "scripts").exists():
        ROOT = ROOT.parent
    elif (ROOT.parent.parent / "scripts").exists():
        ROOT = ROOT.parent.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RUNS_DIR = ROOT / "runs"
RAW_INDEX = ROOT / "data/manifests/epoch_index_yoto_tones.csv"
ZUNA_INDEX = ROOT / "data/manifests/epoch_index_yoto_tones_zuna.csv"

TRAIN_EPOCHS = 5
BATCH_SIZE = 16
LR = 1e-4
DEVICE = "cpu"
CHECKPOINT = ""  # Optional path to a pretrained LaBraM checkpoint
PLOT = True

In [ ]:
from scripts.compare_labram_runs import compare_labram_runs
from scripts.train_labram import train_labram

RUNS = [
    ("raw", RAW_INDEX, RUNS_DIR / "labram_metrics_raw.json"),
    ("raw_zuna", ZUNA_INDEX, RUNS_DIR / "labram_metrics_raw_zuna.json"),
]

for name, index_path, out_path in RUNS:
    if not index_path.exists():
        print(f"Skip {name}: {index_path} not found")
        continue
    try:
        train_labram(
            epochs=TRAIN_EPOCHS,
            batch_size=BATCH_SIZE,
            lr=LR,
            device=DEVICE,
            checkpoint=CHECKPOINT,
            epoch_index=index_path,
            out_metrics=out_path,
        )
    except ValueError as exc:
        print(f"Skip {name}: {exc}")

compare_labram_runs(runs_dir=RUNS_DIR, plot=PLOT)